# XGBoost Two‑Stage Model – 30‑Minute Waste Forecast

This notebook trains an XGBoost **two‑stage model** (classifier + regressor) for each canteen section.
- Classifier predicts whether waste will occur (handles zero inflation).
- Regressor predicts the amount when waste is expected.
- Class imbalance is handled via `scale_pos_weight`.

Data is aggregated to 30‑minute intervals. Features include lags, rolling statistics, and cyclical time encodings.

Saves models and plots to `deployment_models/`.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix
)
import joblib
from google.colab import drive

warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Mount Google Drive and set the working directory
drive.mount('/content/drive', force_remount=True)
try:
    os.chdir('/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence')
    print('Directory changed to project folder')
except OSError:
    print("Error: Could not change directory. Please check the path.")

## Configuration

We set all the parameters for data processing and model training. Changing these values may affect the results.

In [ ]:
DATA_PATH   = 'data/food_waste_augmented.csv'
MODEL_DIR   = 'deployment_models'
MODEL_NAME  = 'XGBoost'

FREQ        = '30min'
ALIGN_SECTIONS = 'union'          # 'common' drops missing, 'union' fills with 0
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.1
TEST_RATIO  = 0.2
LOOKBACK    = 24
MIN_SAMPLES = 10

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Aggregation frequency: {FREQ}")
print(f"Model: {MODEL_NAME}")

## Load and Aggregate Data

We read the waste data, resample it to 30‑minute intervals, and sum the waste per canteen section. Then we pivot the table so that each section becomes a separate column.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
daily_section = (
    df.set_index('Date')
      .groupby([pd.Grouper(freq=FREQ), 'Canteen_Section'])['Waste_Weight_kg']
      .sum()
      .reset_index()
      .rename(columns={'Waste_Weight_kg': 'Total_Waste_kg'})
)

In [ ]:
daily_wide = (
    daily_section
    .pivot(index='Date', columns='Canteen_Section', values='Total_Waste_kg')
    .sort_index()
)

if ALIGN_SECTIONS == 'common':
    daily_wide = daily_wide.dropna()
else:
    daily_wide = daily_wide.fillna(0)

sections = daily_wide.columns.tolist()
print(f"Sections: {sections}")

## Feature Engineering

We create features for each section: time‑based encodings, lagged waste values, rolling statistics, and exponentially weighted means.

In [ ]:
def create_features_for_series(series: pd.Series) -> pd.DataFrame:
    """Create lag, rolling, and cyclical time features from a waste series."""
    df_ml = pd.DataFrame(index=series.index)
    df_ml['y'] = series.values
    df_ml['hour']      = df_ml.index.hour
    df_ml['minute']    = df_ml.index.minute
    df_ml['dayofweek'] = df_ml.index.dayofweek
    df_ml['day']       = df_ml.index.day
    df_ml['month']     = df_ml.index.month
    df_ml['quarter']   = df_ml.index.quarter
    df_ml['weekend']   = (df_ml.index.dayofweek >= 5).astype(int)

    df_ml['hour_sin'] = np.sin(2 * np.pi * df_ml['hour'] / 24)
    df_ml['hour_cos'] = np.cos(2 * np.pi * df_ml['hour'] / 24)
    df_ml['dow_sin'] = np.sin(2 * np.pi * df_ml['dayofweek'] / 7)
    df_ml['dow_cos'] = np.cos(2 * np.pi * df_ml['dayofweek'] / 7)

    for lag in [1, 2, 3, 48, 96]:
        df_ml[f'lag_{lag}'] = df_ml['y'].shift(lag)

    shifted = df_ml['y'].shift(1)
    df_ml['rolling_mean_48'] = shifted.rolling(48).mean()
    df_ml['rolling_std_48']  = shifted.rolling(48).std()
    df_ml['rolling_min_48']  = shifted.rolling(48).min()
    df_ml['rolling_max_48']  = shifted.rolling(48).max()
    df_ml['ewm_mean_48']     = shifted.ewm(span=48).mean()

    df_ml.dropna(inplace=True)
    return df_ml

In [ ]:
# Build feature DataFrames for each section
feature_dfs = {}
for sec in sections:
    feat_df = create_features_for_series(daily_wide[sec])
    feature_dfs[sec] = feat_df

### Align Sections

We align all sections to a common time index. Missing values are filled with 0 (meaning no waste in that period for that section).

In [ ]:
all_idx = pd.DatetimeIndex([])
for df in feature_dfs.values():
    all_idx = all_idx.union(df.index)
all_idx = all_idx.sort_values()
for sec in sections:
    feature_dfs[sec] = feature_dfs[sec].reindex(all_idx).fillna(0)

print(f"Total periods after alignment: {len(all_idx)}")

## Split into Train, Validation, Test

We divide the time series chronologically. The first 70% is for training, next 10% for validation, and the last 20% for testing.

In [ ]:
ref_index = feature_dfs[sections[0]].index
n_total   = len(ref_index)
n_test = max(1, int(n_total * TEST_RATIO))
n_val  = max(1, int(n_total * VAL_RATIO))
n_train = n_total - n_val - n_test

train_indices = ref_index[:n_train]
val_indices   = ref_index[n_train:n_train + n_val]
test_indices  = ref_index[n_train + n_val:]

train_mask = ref_index.isin(train_indices)
val_mask   = ref_index.isin(val_indices)
test_mask  = ref_index.isin(test_indices)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

## Helper Functions

We define functions to compute metrics, plot confusion matrices, residuals, feature importance, and prediction time series.

In [ ]:
def compute_regression_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R² for regression."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

In [ ]:
def compute_classification_metrics(y_true, y_pred, y_prob=None):
    """Compute classification metrics."""
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        metrics['AUC'] = roc_auc_score(y_true, y_prob)
    return metrics

In [ ]:
def plot_confusion_matrix(y_true, y_pred, section, save_path=None):
    """Plot confusion matrix for the classifier."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Waste', 'Waste'],
                yticklabels=['No Waste', 'Waste'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - XGBoost (Section {section})')
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_residuals(y_true, y_pred, section, save_path=None):
    """Plot residuals vs predicted and histogram."""
    residuals = y_true - y_pred
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    axes[0].scatter(y_pred, residuals, alpha=0.5)
    axes[0].axhline(y=0, color='r', linestyle='--')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
    axes[1].hist(residuals, bins=50, edgecolor='black')
    axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Frequency'); axes[1].set_title('Distribution of Residuals')
    plt.suptitle(f'XGBoost (regressor) - Section {section}')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_feature_importance(model, feature_names, section, save_path=None, max_features=20):
    """Plot feature importance from XGBoost regressor."""
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[-max_features:]
        plt.figure(figsize=(10,6))
        plt.barh(range(len(indices)), importances[indices], align='center')
        plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
        plt.xlabel('Feature Importance')
        plt.title(f'Feature Importance - XGBoost (Section {section})')
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
def plot_predictions(y_true, y_pred, dates, section, max_points=200, save_path=None):
    """Plot predicted vs actual waste over time."""
    if len(dates) > max_points:
        step = len(dates) // max_points
        idx = range(0, len(dates), step)
        dates = dates[idx]; y_true = y_true[idx]; y_pred = y_pred[idx]
    plt.figure(figsize=(12,5))
    plt.plot(dates, y_true, label='Actual', marker='o', markersize=3, linestyle='-', linewidth=1)
    plt.plot(dates, y_pred, label='Predicted', marker='x', markersize=3, linestyle='--', linewidth=1)
    plt.xlabel('Date'); plt.ylabel('Waste (kg)')
    plt.title(f'XGBoost Predictions vs Actual - Section {section}')
    plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## Train a Separate Two‑Stage XGBoost for Each Canteen Section

We loop through each section, extract features, and train two models:
1. **Classifier** – predicts if waste will occur (y > 0).
2. **Regressor** – predicts the waste amount, trained only on positive samples.

Finally, we combine predictions and evaluate both stages.

In [ ]:
all_metrics = []

In [ ]:
for sec in sections:
    print(f"\n{'='*60}\nTraining XGBoost for section {sec}\n{'='*60}")

    df_ml = feature_dfs[sec]
    X = df_ml.drop('y', axis=1)
    y = df_ml['y']

    X_train = X[train_mask]; y_train = y[train_mask]
    X_val   = X[val_mask];   y_val   = y[val_mask]
    X_test  = X[test_mask];  y_test  = y[test_mask]

    if len(X_train) == 0:
        print(f"  Skipping section {sec} – training set empty.")
        continue

    feature_columns = X.columns.tolist()

In [ ]:
# --- Train Classifier ---
y_train_cls = (y_train > 0).astype(int)
y_val_cls   = (y_val > 0).astype(int)
neg = (y_train_cls == 0).sum()
pos = (y_train_cls == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0

cls = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.1, max_depth=4,
    scale_pos_weight=scale_pos_weight, random_state=42,
    early_stopping_rounds=10, eval_metric='logloss', use_label_encoder=False
)
cls.fit(X_train, y_train_cls, eval_set=[(X_val, y_val_cls)], verbose=False)

y_prob = cls.predict_proba(X_test)[:, 1]
y_pred_cls = cls.predict(X_test)

In [ ]:
# --- Train Regressor (only on positive samples) ---
mask_train_pos = y_train > 0
mask_val_pos   = y_val > 0

if mask_train_pos.sum() >= 10:
    reg = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        random_state=42, early_stopping_rounds=10, eval_metric='rmse'
    )
    reg.fit(
        X_train[mask_train_pos], y_train[mask_train_pos],
        eval_set=[(X_val[mask_val_pos], y_val[mask_val_pos])],
        verbose=False
    )
    reg_pred = reg.predict(X_test)
    median_pos = None
else:
    print(f"  Warning: only {mask_train_pos.sum()} positive training samples. Using median.")
    median_pos = y_train[mask_train_pos].median() if mask_train_pos.sum() > 0 else 0
    reg_pred = np.full(len(X_test), median_pos)
    reg = None

In [ ]:
# --- Combine Predictions ---
final_pred = np.where(y_pred_cls == 1, reg_pred, 0)

In [ ]:
# --- Metrics ---
cls_metrics = compute_classification_metrics((y_test > 0).astype(int), y_pred_cls, y_prob)
mask_test_pos = y_test > 0
if mask_test_pos.sum() > 0 and reg is not None:
    reg_metrics = compute_regression_metrics(y_test[mask_test_pos], reg_pred[mask_test_pos])
else:
    reg_metrics = {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
overall_metrics = compute_regression_metrics(y_test.values, final_pred)

print("\n--- Classification ---")
for k,v in cls_metrics.items(): print(f"  {k}: {v:.4f}")
print("\n--- Regression (positive test samples) ---")
for k,v in reg_metrics.items(): print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: N/A")
print("\n--- Overall ---")
for k,v in overall_metrics.items(): print(f"  {k}: {v:.4f}")

all_metrics.append({
    'Section': sec, 'Model': MODEL_NAME,
    **{f'cls_{k}': v for k,v in cls_metrics.items()},
    **{f'reg_{k}': v for k,v in reg_metrics.items()},
    **{f'overall_{k}': v for k,v in overall_metrics.items()}
})

In [ ]:
# --- Save Models and Metadata ---
artifacts = {
    'section': sec,
    'model_name': MODEL_NAME,
    'two_stage': True,
    'classifier': cls,
    'regressor': reg,
    'fallback_median': median_pos,
    'feature_columns': feature_columns,
    'lookback': LOOKBACK,
    'train_date_range': [train_indices.min().isoformat(), train_indices.max().isoformat()],
    'val_date_range': [val_indices.min().isoformat(), val_indices.max().isoformat()],
    'test_date_range': [test_indices.min().isoformat(), test_indices.max().isoformat()]
}
save_path = f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}.joblib'
joblib.dump(artifacts, save_path)
print(f"Saved model to {save_path}")

In [ ]:
# --- Plots ---
plot_confusion_matrix((y_test > 0).astype(int), y_pred_cls, sec,
                        save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_confusion.png')
if mask_test_pos.sum() > 0 and reg is not None:
    plot_residuals(y_test[mask_test_pos], reg_pred[mask_test_pos], sec,
                    save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_residuals.png')
if reg is not None:
    plot_feature_importance(reg, feature_columns, sec,
                            save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_feature_importance.png')
plot_predictions(y_test.values, final_pred, test_indices, sec,
                    save_path=f'{MODEL_DIR}/section_{sec}_{MODEL_NAME}_predictions.png')

## Save Summary of Results

After training all sections, we collect the performance metrics into a CSV file for later comparison.

In [ ]:
if all_metrics:
    summary_df = pd.DataFrame(all_metrics)
    summary_df.to_csv(f'{MODEL_DIR}/test_metrics_{MODEL_NAME}.csv', index=False)
    print("\nSummary saved.")
else:
    print("No metrics collected.")